# Keyword Price Outliers With Name Blacklist Filter

`keyword_price_outliers.ipynb` 흐름을 복사하되, 가격 이상치 분석 전에 크롤링 단계에서 제거할 이름 블랙리스트 필터를 먼저 적용합니다.

블랙리스트 필터 기준:
- `DROP_NAME_KEYWORDS`에 제외할 단어를 입력합니다.
- 크롤링된 `name`에 해당 단어가 포함되면 드랍합니다.
- 예: 케이스, 필름, 강화유리, 맥세이프, 충전기 등 본체 가격 통계를 흐리는 액세서리/구매글/홍보 문구

이 노트북은 CSV 저장보다 결과 확인용 DataFrame 출력에 초점을 둡니다.


In [20]:
from pathlib import Path
import re

import pandas as pd


def resolve_analysis_dir() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == "analysis":
        return cwd

    candidate = cwd / "code/backend/src/main/python/analysis"
    if candidate.exists():
        return candidate

    return Path(__file__).resolve().parent if "__file__" in globals() else cwd


ANALYSIS_DIR = resolve_analysis_dir()
PYTHON_DIR = ANALYSIS_DIR.parent
CRAWLING_RESULTS_DIR = PYTHON_DIR / "crawling" / "results"

candidates = sorted(
    CRAWLING_RESULTS_DIR.glob("통합조회_전체_no_filter_*.csv"),
    key=lambda path: path.stat().st_mtime,
)

if not candidates:
    raise FileNotFoundError(f"no-filter 크롤링 CSV를 찾을 수 없습니다: {CRAWLING_RESULTS_DIR}")

csv_path = candidates[-1]
csv_path


WindowsPath('C:/project/kdtproject/kdtproject/code/backend/src/main/python/crawling/results/통합조회_전체_no_filter_20260522_1106.csv')

In [21]:
# 여기에 크롤링 단계에서 제외하고 싶은 name 포함 키워드를 추가하세요.
DROP_NAME_KEYWORDS = [
    "삽니다",
    "구매합니다",
    "매입",
    "대여",
    "교환",
    "렌탈",
    "매입",
    "대여",
]


def normalize_search_text(value):
    text = str(value or "").lower()
    text = re.sub(r"(?<=[a-z0-9])plus\b", " 플러스", text)
    text = re.sub(r"\bplus\b|[+＋]", " 플러스 ", text)
    text = re.sub(r"\bpro\b", " 프로 ", text)
    text = re.sub(r"\bmax\b", " 맥스 ", text)
    text = re.sub(r"\bultra\b", " 울트라 ", text)
    text = re.sub(r"[^0-9a-z가-힣\s]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def compact_search_text(value):
    return re.sub(r"\s+", "", normalize_search_text(value))


def build_drop_name_keywords(keywords):
    drop_keywords = []
    seen = set()

    for keyword in keywords:
        original = str(keyword or "").strip()
        normalized = normalize_search_text(original)
        compact = compact_search_text(original)
        key = (normalized, compact)

        if not normalized or key in seen:
            continue

        seen.add(key)
        drop_keywords.append(
            {
                "original": original,
                "normalized": normalized,
                "compact": compact,
            }
        )

    return drop_keywords


DROP_NAME_KEYWORD_MATCHERS = build_drop_name_keywords(DROP_NAME_KEYWORDS)
NORMALIZED_DROP_NAME_KEYWORDS = [keyword["normalized"] for keyword in DROP_NAME_KEYWORD_MATCHERS]


def matched_drop_keywords(name):
    raw_name = str(name or "").lower()
    normalized_name = normalize_search_text(name)
    compact_name = compact_search_text(name)

    matched_keywords = []
    for keyword in DROP_NAME_KEYWORD_MATCHERS:
        raw_keyword = keyword["original"].lower()
        normalized_keyword = keyword["normalized"]
        compact_keyword = keyword["compact"]

        if (
            raw_keyword in raw_name
            or normalized_keyword in normalized_name
            or compact_keyword in compact_name
        ):
            matched_keywords.append(keyword["original"])

    return matched_keywords


In [22]:
df = pd.read_csv(csv_path, encoding="utf-8-sig")

working_df = df.copy()
working_df["price_numeric"] = pd.to_numeric(working_df["price"], errors="coerce")
working_df = working_df.dropna(subset=["keyword", "name", "price_numeric"])
working_df = working_df[working_df["price_numeric"] > 0].copy()
working_df["price_numeric"] = working_df["price_numeric"].astype(int)

working_df["matched_drop_keywords"] = working_df["name"].apply(matched_drop_keywords)
working_df["name_blacklist_drop"] = working_df["matched_drop_keywords"].apply(bool)
working_df["matched_drop_keywords_text"] = working_df["matched_drop_keywords"].apply(lambda values: " | ".join(values))

filtered_df = working_df[~working_df["name_blacklist_drop"]].copy()
dropped_filter_df = working_df[working_df["name_blacklist_drop"]].copy()

filter_overview_df = pd.DataFrame([
    {
        "source_file": csv_path.name,
        "valid_price_rows_before_filter": len(working_df),
        "rows_after_name_blacklist_filter": len(filtered_df),
        "removed_by_name_blacklist_filter": len(dropped_filter_df),
        "name_blacklist_keep_rate": len(filtered_df) / len(working_df) if len(working_df) else 0,
        "keyword_count_before_filter": working_df["keyword"].nunique(),
        "keyword_count_after_filter": filtered_df["keyword"].nunique(),
        "drop_keyword_count": len(NORMALIZED_DROP_NAME_KEYWORDS),
    }
])

filter_overview_df


,source_file,valid_price_rows_before_filter,rows_after_name_blacklist_filter,removed_by_name_blacklist_filter,name_blacklist_keep_rate,keyword_count_before_filter,keyword_count_after_filter,drop_keyword_count
0,통합조회_전체_no_filter_20260522_1106.csv,24988,24249,739,0.970426,20,20,6


In [23]:
filter_keyword_summary_df = working_df.groupby("keyword").agg(
    before_count=("keyword", "count"),
    dropped_by_name_blacklist=("name_blacklist_drop", "sum"),
).reset_index()
filter_keyword_summary_df["after_name_blacklist_count"] = (
    filter_keyword_summary_df["before_count"] - filter_keyword_summary_df["dropped_by_name_blacklist"]
)
filter_keyword_summary_df["name_blacklist_drop_rate"] = (
    filter_keyword_summary_df["dropped_by_name_blacklist"] / filter_keyword_summary_df["before_count"]
)
filter_keyword_summary_df = filter_keyword_summary_df.sort_values(
    ["dropped_by_name_blacklist", "before_count"], ascending=[False, False]
)

filter_keyword_summary_df


,keyword,before_count,dropped_by_name_blacklist,after_name_blacklist_count,name_blacklist_drop_rate
16,스텔라이브,1868,340,1528,0.182013
4,갤럭시,10414,143,10271,0.013732
0,5090,309,47,262,0.152104
6,골드바,526,46,480,0.087452
14,세이코,1986,42,1944,0.021148
5,게이밍 노트북,885,25,860,0.028249
13,붉은사막,392,22,370,0.056122
2,ps5,1018,14,1004,0.013752
19,핫토이,1812,9,1803,0.004967
15,센티넬,206,9,197,0.043689


In [ ]:
# 1차 name 블랙리스트 필터 적용 후 keyword별 저가 데이터 조회
LOW_PRICE_PER_KEYWORD_LIMIT = 30
LOW_PRICE_KEYWORD = None  # 예: "갤럭시", "5090". 전체 keyword 조회는 None

low_price_review_base_df = filtered_df.copy()

if LOW_PRICE_KEYWORD:
    low_price_review_base_df = low_price_review_base_df[
        low_price_review_base_df["keyword"].astype(str).str.contains(LOW_PRICE_KEYWORD, case=False, na=False)
    ]

low_price_review_df = (
    low_price_review_base_df.sort_values(["keyword", "price_numeric", "platform", "pid"])
    .groupby("keyword", group_keys=False)
    .head(LOW_PRICE_PER_KEYWORD_LIMIT)
    .copy()
)
low_price_review_df["low_price_rank_in_keyword"] = (
    low_price_review_df.groupby("keyword").cumcount() + 1
)

low_price_review_df = low_price_review_df[
    [
        "keyword",
        "low_price_rank_in_keyword",
        "platform",
        "pid",
        "name",
        "price_numeric",
        "status",
        "link",
    ]
].sort_values(["keyword", "low_price_rank_in_keyword"])

print(
    f"조회 keyword 수: {low_price_review_df['keyword'].nunique():,}, "
    f"조회 row 수: {len(low_price_review_df):,}"
)

with pd.option_context("display.max_rows", len(low_price_review_df), "display.max_colwidth", 120):
    display(low_price_review_df)


In [24]:
dropped_filter_df[[
    "keyword",
    "platform",
    "pid",
    "name",
    "price_numeric",
    "matched_drop_keywords_text",
    "status",
    "link",
]].sort_values(["keyword", "platform", "price_numeric"]).head(100)


,keyword,platform,pid,name,price_numeric,matched_drop_keywords_text,status,link
5713,5090,번개장터,397111572,RTX5090 대량 매입합니다! 모델따지지않고 A/S기간 따지지않고 작동,550,매입,판매중,https://m.bunjang.co.kr/products/397111572
5714,5090,번개장터,368439995,RTX5090삽니다! 삽니다 !,550,삽니다,판매중,https://m.bunjang.co.kr/products/368439995
5704,5090,번개장터,390058761,(가격인하) Rtx5080 만기인수형 장기렌탈,132900,렌탈,판매중,https://m.bunjang.co.kr/products/390058761
5746,5090,번개장터,409019934,슈프림(저)=추금+5090 하~중급기 교환합니다,300000,교환,판매중,https://m.bunjang.co.kr/products/409019934
5705,5090,번개장터,389838065,Rtx5090 만기인수형 장기렌탈,360000,렌탈,판매중,https://m.bunjang.co.kr/products/389838065
...,...,...,...,...,...,...,...,...
13499,갤럭시,번개장터,281560620,[대여] 갤럭시S23 울트라,19000,대여,판매중,https://m.bunjang.co.kr/products/281560620
13118,갤럭시,번개장터,381028666,갤럭시 울트라 대여해드립니다 라이즈 박지훈 앤더블,20000,대여,판매중,https://m.bunjang.co.kr/products/381028666
13259,갤럭시,번개장터,409443535,갤럭시 울트라 26 대여 받아요,20000,대여,판매중,https://m.bunjang.co.kr/products/409443535
14771,갤럭시,번개장터,282680861,"[프로상점,대여]갤럭시s23울트라 (256/512기가)",20000,대여,판매중,https://m.bunjang.co.kr/products/282680861


In [25]:
LOW_PRICE_MEDIAN_RATIO = 0.2

keyword_stats_df = (
    filtered_df.groupby("keyword")["price_numeric"]
    .agg(
        item_count="count",
        min_price="min",
        q1=lambda values: values.quantile(0.25),
        median_price="median",
        q3=lambda values: values.quantile(0.75),
        max_price="max",
        average_price="mean",
    )
    .reset_index()
)

keyword_stats_df["iqr"] = keyword_stats_df["q3"] - keyword_stats_df["q1"]
keyword_stats_df["iqr_lower_bound"] = (keyword_stats_df["q1"] - 1.5 * keyword_stats_df["iqr"]).clip(lower=0)
keyword_stats_df["median_ratio_lower_bound"] = keyword_stats_df["median_price"] * LOW_PRICE_MEDIAN_RATIO
keyword_stats_df["lower_bound"] = keyword_stats_df[["iqr_lower_bound", "median_ratio_lower_bound"]].max(axis=1)
keyword_stats_df["upper_bound"] = keyword_stats_df["q3"] + 1.5 * keyword_stats_df["iqr"]

keyword_stats_df.sort_values(["item_count", "keyword"], ascending=[False, True])


,keyword,item_count,min_price,q1,median_price,q3,max_price,average_price,iqr,iqr_lower_bound,median_ratio_lower_bound,lower_bound,upper_bound
4,갤럭시,10271,500,85000.0,159000.0,328500.0,7500000,2.735680e+05,243500.0,0.0,31800.0,31800.0,693750.0
14,세이코,1944,168,140000.0,244500.0,500000.0,12250000,6.056621e+05,360000.0,0.0,48900.0,48900.0,1040000.0
19,핫토이,1803,100,113000.0,220000.0,350000.0,999999999,9.261663e+05,237000.0,0.0,44000.0,44000.0,705500.0
16,스텔라이브,1528,500,12000.0,30000.0,70000.0,999999999,3.135779e+06,58000.0,0.0,6000.0,6000.0,157000.0
7,던스,1237,1,35000.0,60000.0,105000.0,4444444,9.047857e+04,70000.0,0.0,12000.0,12000.0,210000.0
2,ps5,1004,1,23500.0,45000.0,117325.0,90105000,2.584678e+05,93825.0,0.0,9000.0,9000.0,258062.5
8,만년필,990,100,45000.0,150000.0,441500.0,99999999,8.327042e+05,396500.0,0.0,30000.0,30000.0,1036250.0
12,바이씨니,950,10000,50000.0,70000.0,105000.0,999999,8.688152e+04,55000.0,0.0,14000.0,14000.0,187500.0
5,게이밍 노트북,860,500,450000.0,900000.0,1500000.0,9500000,1.131848e+06,1050000.0,0.0,180000.0,180000.0,3075000.0
18,제라늄,799,1111,8000.0,12345.0,25000.0,420000,2.114209e+04,17000.0,0.0,2469.0,2469.0,50500.0


In [26]:
outlier_df = filtered_df.merge(
    keyword_stats_df[[
        "keyword",
        "q1",
        "median_price",
        "q3",
        "iqr",
        "iqr_lower_bound",
        "median_ratio_lower_bound",
        "lower_bound",
        "upper_bound",
    ]],
    on="keyword",
    how="left",
)

outlier_df["outlier_type"] = ""
outlier_df.loc[outlier_df["price_numeric"] < outlier_df["lower_bound"], "outlier_type"] = "low"
outlier_df.loc[outlier_df["price_numeric"] > outlier_df["upper_bound"], "outlier_type"] = "high"
outlier_df = outlier_df[outlier_df["outlier_type"] != ""].copy()
outlier_df["price_to_median_ratio"] = outlier_df["price_numeric"] / outlier_df["median_price"]

outlier_df[[
    "keyword",
    "platform",
    "pid",
    "name",
    "price_numeric",
    "outlier_type",
    "lower_bound",
    "median_price",
    "upper_bound",
    "price_to_median_ratio",
    "status",
    "link",
]].sort_values(["keyword", "outlier_type", "price_numeric"]).head(100)


,keyword,platform,pid,name,price_numeric,outlier_type,lower_bound,median_price,upper_bound,price_to_median_ratio,status,link
5512,5090,중고나라,228457956,5090 3개 일괄 판매합니다,16500000,high,1200000.0,6000000.0,13625000.0,2.750000,거래완료,https://web.joongna.com/product/228457956
5435,5090,중고나라,223733140,사기 5070 5080 4090 5090 9950 9800 p41 p51 990 9910,112,low,1200000.0,6000000.0,13625000.0,0.000019,판매중,https://web.joongna.com/product/223733140
5461,5090,중고나라,228724580,ROG ASTRAL 5090,550,low,1200000.0,6000000.0,13625000.0,0.000092,거래완료,https://web.joongna.com/product/228724580
5483,5090,중고나라,228509681,9800X3D 5090 엔드급 초고사양 하이엔드 컴퓨터 본체 PC,1004,low,1200000.0,6000000.0,13625000.0,0.000167,거래완료,https://web.joongna.com/product/228509681
5299,5090,번개장터,408316423,"9800x3d, B850m wifi, rtx 5080 화이트 완본체",1111,low,1200000.0,6000000.0,13625000.0,0.000185,예약중,https://m.bunjang.co.kr/products/408316423
...,...,...,...,...,...,...,...,...,...,...,...,...
5159,a7c2,번개장터,370271048,아반테MD ECU 39110-2BAA7 판매,90000,low,360000.0,1800000.0,4953750.0,0.050000,판매중,https://m.bunjang.co.kr/products/370271048
5167,a7c2,중고나라,228713159,소니 a7c2 a7cr gp-x2 gpx2 미사용 확장그립팝니다,90000,low,360000.0,1800000.0,4953750.0,0.050000,판매중,https://web.joongna.com/product/228713159
5259,a7c2,중고나라,228402278,소니 A7C 및 A7C2용 확장그립 (GP-X2) 팝니다.,90000,low,360000.0,1800000.0,4953750.0,0.050000,거래완료,https://web.joongna.com/product/228402278
5199,a7c2,중고나라,228649415,"소니 GP-X2 A7CR, A7C2 확장 그립",99000,low,360000.0,1800000.0,4953750.0,0.055000,판매중,https://web.joongna.com/product/228649415


In [27]:
# 가격 이상치 name 분석용 조회 셀
# 필요하면 아래 조건만 바꿔서 다시 실행하세요.
OUTLIER_KEYWORD = None  # 예: "갤럭시", "5090". 전체 조회는 None
OUTLIER_TYPE = None  # "low", "high" 또는 전체 조회는 None
NAME_CONTAINS = ""  # name에 포함된 단어로 추가 검색. 전체 조회는 빈 문자열
OUTLIER_LIMIT = 300

price_outlier_review_df = outlier_df.copy()
price_outlier_review_df["price_gap_from_median"] = (
    price_outlier_review_df["price_numeric"] - price_outlier_review_df["median_price"]
)
price_outlier_review_df["outlier_severity"] = price_outlier_review_df.apply(
    lambda row: row["median_price"] / row["price_numeric"]
    if row["outlier_type"] == "low"
    else row["price_to_median_ratio"],
    axis=1,
)

if OUTLIER_KEYWORD:
    price_outlier_review_df = price_outlier_review_df[
        price_outlier_review_df["keyword"].astype(str).str.contains(OUTLIER_KEYWORD, case=False, na=False)
    ]

if OUTLIER_TYPE:
    price_outlier_review_df = price_outlier_review_df[
        price_outlier_review_df["outlier_type"].eq(OUTLIER_TYPE)
    ]

if NAME_CONTAINS:
    price_outlier_review_df = price_outlier_review_df[
        price_outlier_review_df["name"].astype(str).str.contains(NAME_CONTAINS, case=False, na=False)
    ]

price_outlier_review_df = price_outlier_review_df[
    [
        "keyword",
        "outlier_type",
        "platform",
        "pid",
        "name",
        "price_numeric",
        "median_price",
        "price_gap_from_median",
        "price_to_median_ratio",
        "outlier_severity",
        "lower_bound",
        "upper_bound",
        "status",
        "link",
    ]
].sort_values(
    ["keyword", "outlier_type", "outlier_severity"],
    ascending=[True, True, False],
)

print(f"조회된 가격 이상치 수: {len(price_outlier_review_df):,}")
price_outlier_review_df.head(OUTLIER_LIMIT)


조회된 가격 이상치 수: 4,948


,keyword,outlier_type,platform,pid,name,price_numeric,median_price,price_gap_from_median,price_to_median_ratio,outlier_severity,lower_bound,upper_bound,status,link
5512,5090,high,중고나라,228457956,5090 3개 일괄 판매합니다,16500000,6000000.0,10500000.0,2.750000,2.750000,1200000.0,13625000.0,거래완료,https://web.joongna.com/product/228457956
5435,5090,low,중고나라,223733140,사기 5070 5080 4090 5090 9950 9800 p41 p51 990 9910,112,6000000.0,-5999888.0,0.000019,53571.428571,1200000.0,13625000.0,판매중,https://web.joongna.com/product/223733140
5461,5090,low,중고나라,228724580,ROG ASTRAL 5090,550,6000000.0,-5999450.0,0.000092,10909.090909,1200000.0,13625000.0,거래완료,https://web.joongna.com/product/228724580
5483,5090,low,중고나라,228509681,9800X3D 5090 엔드급 초고사양 하이엔드 컴퓨터 본체 PC,1004,6000000.0,-5998996.0,0.000167,5976.095618,1200000.0,13625000.0,거래완료,https://web.joongna.com/product/228509681
5299,5090,low,번개장터,408316423,"9800x3d, B850m wifi, rtx 5080 화이트 완본체",1111,6000000.0,-5998889.0,0.000185,5400.540054,1200000.0,13625000.0,예약중,https://m.bunjang.co.kr/products/408316423
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6153,ps5,high,번개장터,409257928,Ps5 vr2 판매합니다.,300000,45000.0,255000.0,6.666667,6.666667,9000.0,258062.5,판매중,https://m.bunjang.co.kr/products/409257928
6262,ps5,high,번개장터,408932871,ps4와 ps5로 출시 된 전 아틀리에 시리즈 택포,300000,45000.0,255000.0,6.666667,6.666667,9000.0,258062.5,판매중,https://m.bunjang.co.kr/products/408932871
6327,ps5,high,번개장터,408146928,팰리세이드 LX2 오무기어(스티어링 기어) MDPS 57700 S8050,300000,45000.0,255000.0,6.666667,6.666667,9000.0,258062.5,판매중,https://m.bunjang.co.kr/products/408146928
6378,ps5,high,번개장터,406697954,중고타이어 255 35 19 미쉐린 ps5 두짝 2553519,300000,45000.0,255000.0,6.666667,6.666667,9000.0,258062.5,판매중,https://m.bunjang.co.kr/products/406697954


In [28]:
outlier_summary_df = (
    outlier_df.groupby(["keyword", "outlier_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for column in ["low", "high"]:
    if column not in outlier_summary_df.columns:
        outlier_summary_df[column] = 0

outlier_summary_df["total_outlier_count"] = outlier_summary_df["low"] + outlier_summary_df["high"]
outlier_summary_df = outlier_summary_df.merge(
    keyword_stats_df[["keyword", "item_count", "min_price", "median_price", "max_price", "lower_bound", "upper_bound"]],
    on="keyword",
    how="left",
)
outlier_summary_df["outlier_rate"] = outlier_summary_df["total_outlier_count"] / outlier_summary_df["item_count"]

outlier_overview_df = outlier_summary_df[[
    "keyword",
    "item_count",
    "low",
    "high",
    "total_outlier_count",
    "outlier_rate",
    "min_price",
    "median_price",
    "max_price",
    "lower_bound",
    "upper_bound",
]].sort_values(["total_outlier_count", "outlier_rate"], ascending=[False, False])

overall_outlier_overview_df = pd.DataFrame([
    {
        "source_file": csv_path.name,
        "valid_price_rows_before_filter": len(working_df),
        "rows_after_name_blacklist_filter": len(filtered_df),
        "removed_by_name_blacklist_filter": len(dropped_filter_df),
        "name_blacklist_keep_rate": len(filtered_df) / len(working_df) if len(working_df) else 0,
        "outlier_rows_after_name_blacklist_filter": len(outlier_df),
        "outlier_rate_after_name_blacklist_filter": len(outlier_df) / len(filtered_df) if len(filtered_df) else 0,
        "low_outlier_rows": int((outlier_df["outlier_type"] == "low").sum()) if len(outlier_df) else 0,
        "high_outlier_rows": int((outlier_df["outlier_type"] == "high").sum()) if len(outlier_df) else 0,
        "low_price_median_ratio": LOW_PRICE_MEDIAN_RATIO,
    }
])

display(overall_outlier_overview_df)
outlier_overview_df


,source_file,valid_price_rows_before_filter,rows_after_name_blacklist_filter,removed_by_name_blacklist_filter,name_blacklist_keep_rate,outlier_rows_after_name_blacklist_filter,outlier_rate_after_name_blacklist_filter,low_outlier_rows,high_outlier_rows,low_price_median_ratio
0,통합조회_전체_no_filter_20260522_1106.csv,24988,24249,739,0.970426,4948,0.20405,2801,2147,0.2


,keyword,item_count,low,high,total_outlier_count,outlier_rate,min_price,median_price,max_price,lower_bound,upper_bound
4,갤럭시,10271,1408,931,2339,0.227729,500,159000.0,7500000,31800.0,693750.0
14,세이코,1944,144,239,383,0.197016,168,244500.0,12250000,48900.0,1040000.0
16,스텔라이브,1528,202,165,367,0.240183,500,30000.0,999999999,6000.0,157000.0
19,핫토이,1803,257,106,363,0.201331,100,220000.0,999999999,44000.0,705500.0
8,만년필,990,158,120,278,0.280808,100,150000.0,99999999,30000.0,1036250.0
2,ps5,1004,29,180,209,0.208167,1,45000.0,90105000,9000.0,258062.5
6,골드바,480,106,90,196,0.408333,250,817000.0,999999999,163400.0,1973750.0
7,던스,1237,71,71,142,0.114794,1,60000.0,4444444,12000.0,210000.0
5,게이밍 노트북,860,76,41,117,0.136047,500,900000.0,9500000,180000.0,3075000.0
11,메탈빌드,644,60,19,79,0.122671,1230,326000.0,99999999,65200.0,865000.0


In [29]:
# name 블랙리스트 필터 + 가격 이상치 제거 후 keyword별 가격 요약 DF
outlier_keys = outlier_df[["keyword", "platform", "pid", "price_numeric"]].copy()
outlier_keys["is_outlier"] = True

clean_price_df = filtered_df.merge(
    outlier_keys[["keyword", "platform", "pid", "price_numeric", "is_outlier"]],
    on=["keyword", "platform", "pid", "price_numeric"],
    how="left",
)
clean_price_df["is_outlier"] = clean_price_df["is_outlier"].fillna(False).astype(bool)
clean_price_df = clean_price_df.loc[~clean_price_df["is_outlier"]].copy()

clean_keyword_price_summary_df = (
    clean_price_df.groupby("keyword")["price_numeric"]
    .agg(
        clean_item_count="count",
        clean_min_price="min",
        clean_max_price="max",
        clean_average_price="mean",
        clean_median_price="median",
    )
    .reset_index()
)

filtered_counts_df = filtered_df.groupby("keyword").size().reset_index(name="name_blacklist_filtered_item_count")
outlier_counts_df = outlier_df.groupby("keyword").size().reset_index(name="removed_price_outlier_count")

clean_keyword_price_summary_df = filtered_counts_df.merge(
    outlier_counts_df,
    on="keyword",
    how="left",
).merge(
    clean_keyword_price_summary_df,
    on="keyword",
    how="left",
)
clean_keyword_price_summary_df["removed_price_outlier_count"] = clean_keyword_price_summary_df["removed_price_outlier_count"].fillna(0).astype(int)
clean_keyword_price_summary_df["clean_item_count"] = clean_keyword_price_summary_df["clean_item_count"].fillna(0).astype(int)
clean_keyword_price_summary_df["removed_price_outlier_rate"] = clean_keyword_price_summary_df["removed_price_outlier_count"] / clean_keyword_price_summary_df["name_blacklist_filtered_item_count"]

clean_keyword_price_summary_df = clean_keyword_price_summary_df[[
    "keyword",
    "name_blacklist_filtered_item_count",
    "removed_price_outlier_count",
    "removed_price_outlier_rate",
    "clean_item_count",
    "clean_min_price",
    "clean_max_price",
    "clean_average_price",
    "clean_median_price",
]].sort_values("keyword")

clean_keyword_price_summary_df


,keyword,name_blacklist_filtered_item_count,removed_price_outlier_count,removed_price_outlier_rate,clean_item_count,clean_min_price,clean_max_price,clean_average_price,clean_median_price
0,5090,262,57,0.217557,205,1200000,11000000,7.066006e+06,7375000.0
1,a7c2,196,69,0.352041,127,500000,4600000,2.126567e+06,1950000.0
2,ps5,1004,209,0.208167,795,9000,250000,4.998340e+04,35000.0
3,y700,220,39,0.177273,181,75000,1030000,3.904589e+05,383000.0
4,갤럭시,10271,2339,0.227729,7932,31910,691480,2.155599e+05,165000.0
5,게이밍 노트북,860,117,0.136047,743,180000,3050000,1.045665e+06,950000.0
6,골드바,480,196,0.408333,284,175000,1800000,6.769950e+05,819500.0
7,던스,1237,142,0.114794,1095,12000,210000,7.298848e+04,60000.0
8,만년필,990,278,0.280808,712,30000,1000000,2.386365e+05,150000.0
9,맥미니 m4,93,13,0.139785,80,370000,2300000,1.102788e+06,990000.0
